# Station Heights: Elevation from DOM1

Samples elevation (meters above sea level) for each bike station from the DOM1 GeoTIFF tiles.
Adds `elevation_m` field to `prepared-data/stations.json`.

## Setup & Imports

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt

cwd = Path.cwd()
project_root = cwd if (cwd / "package.json").exists() else cwd.parent.parent
prepared_dir = project_root / "prepared-data"
dom1_dir = project_root / "dom1"

sys.path.insert(0, str(project_root / "data-pipeline"))
from execution_utils import show_execution_banner, write_with_execution_metadata
from elevation_utils import build_tile_index, sample_elevation

print("Project root:", project_root)
print("DOM1 data:", dom1_dir)

out_path = prepared_dir / "stations.json"
_pipeline_start_time = show_execution_banner(out_path)

## Load Stations & Build Tile Index

In [ ]:
with open(out_path, encoding="utf-8") as f:
    stations_file = json.load(f)

stations = stations_file["data"]["stations"]
print(f"Loaded {len(stations)} stations")

tile_index = build_tile_index(dom1_dir)
print(f"Found {len(tile_index)} tiles")
for t in tile_index:
    print(f"  {t['path'].name}: {t['ncols']}x{t['nrows']} px, "
          f"X=[{t['x_min']:.0f}, {t['x_max']:.0f}], Y=[{t['y_min']:.0f}, {t['y_max']:.0f}]")

## Sample Elevation per Station

In [ ]:
missing = []
for s in stations:
    elev = sample_elevation(s["lat"], s["lon"], tile_index)
    s["elevation_m"] = elev
    if elev is None:
        missing.append(s["id"])

elevations = [s["elevation_m"] for s in stations if s["elevation_m"] is not None]
print(f"Sampled elevation for {len(elevations)}/{len(stations)} stations")
if missing:
    print(f"Missing: {missing}")
print(f"Min: {min(elevations):.1f} m, Max: {max(elevations):.1f} m, "
      f"Mean: {sum(elevations)/len(elevations):.1f} m")

## Visualize

In [ ]:
sorted_stations = sorted(
    [s for s in stations if s["elevation_m"] is not None],
    key=lambda s: s["elevation_m"]
)

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(len(sorted_stations)), [s["elevation_m"] for s in sorted_stations], width=1.0)
ax.set_xlabel("Station (sorted by elevation)")
ax.set_ylabel("Elevation (m)")
ax.set_title("Oslo Bysykkel Station Elevations")
plt.tight_layout()
plt.show()

## Write Output

In [ ]:
write_with_execution_metadata(out_path, {"stations": stations}, _pipeline_start_time)
print(f"Wrote {out_path}")